# 02 - PhoBERT Sentiment & Annotation Pipeline

This notebook acts as an execution and inspection runner for the sentiment analysis and article annotation pipeline.

When running on Kaggle, this notebook will:
- Read keys (`GITHUB_TOKEN`, `GEMINI_API_KEY`, etc.) from Kaggle Secrets.
- Clone or pull the active branch of the GitHub repo into `/kaggle/working`.
- Materialize credentials in `.dvc/credentials/key.json` and `.dvc/config.local` for service account access.
- Pull DVC tracking targets (like raw news files).
- Install dependencies and run pipeline stages against the checked-out workspace.

## 0 - Bootstrap

In [ ]:
import base64
import importlib
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

STAGED_ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
IS_KAGGLE = Path('/kaggle').exists()

def run_command(*args: str, cwd: str | Path | None = None, display: str | None = None) -> None:
    command = [str(arg) for arg in args]
    print('$', display or ' '.join(command))
    subprocess.run(command, check=True, cwd=cwd)

def load_json(path: str | Path) -> dict:
    return json.loads(Path(path).read_text(encoding='utf-8'))

def load_kaggle_secret(name: str) -> str | None:
    if not IS_KAGGLE:
        return os.getenv(name)
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.getenv(name)

def parse_json_secret(value: str | None) -> dict | None:
    if not value:
        return None
    candidates = [value]
    try:
        decoded = base64.b64decode(value).decode('utf-8')
        candidates.append(decoded)
    except Exception:
        pass
    for candidate in candidates:
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            continue
    return None

def write_service_account_key(secret_name: str, repo_root: Path) -> Path | None:
    secret_payload = parse_json_secret(load_kaggle_secret(secret_name))
    if not secret_payload:
        return None
    key_path = repo_root / '.dvc' / 'credentials' / 'key.json'
    key_path.parent.mkdir(parents=True, exist_ok=True)
    key_path.write_text(json.dumps(secret_payload, ensure_ascii=False, indent=2), encoding='utf-8')
    return key_path

def configure_dvc_service_account(repo_root: Path, key_path: Path, remote_name: str = 'gdrive') -> Path:
    config_local_path = repo_root / '.dvc' / 'config.local'
    rel_key_path = key_path.relative_to(repo_root / '.dvc')
    config_lines = [
        f'[remote "{remote_name}"]',
        '    gdrive_use_service_account = true',
        f'    gdrive_service_account_json_file_path = {rel_key_path.as_posix()}'
    ]
    config_local_path.write_text('\n'.join(config_lines) + '\n', encoding='utf-8')
    return config_local_path

def github_auth_url(repo_url: str, token: str | None) -> str:
    if not token or not repo_url.startswith('https://github.com/'):
        return repo_url
    return repo_url.replace('https://', f'https://{token}@', 1)

print('Running on Kaggle:', IS_KAGGLE)
print('Staged root     :', STAGED_ROOT)


## 1 - Runtime Configuration

In [ ]:
REPO_GIT_URL = 'https://github.com/tlong-ds/news-sentiment-analysis.git'
REPO_BRANCH = 'main'
REPO_DIR_NAME = 'news-sentiment-analysis'
PULL_REPO_ON_KAGGLE = IS_KAGGLE
INSTALL_PROJECT_REQUIREMENTS = IS_KAGGLE
LOAD_KAGGLE_SECRETS = IS_KAGGLE
GDRIVE_SERVICE_ACCOUNT_SECRET = 'GDRIVE_SERVICE_ACCOUNT_JSON'
SETUP_DVC_SERVICE_ACCOUNT = IS_KAGGLE
DVC_PULL_ON_KAGGLE = IS_KAGGLE
DVC_PULL_REMOTE = 'gdrive'
DVC_PULL_TARGETS = ['data/main/raw.dvc']


## 2 - Kaggle Secrets, Repo Pull, and DVC Bootstrap

In [ ]:
if LOAD_KAGGLE_SECRETS:
    for secret_name in ['GITHUB_TOKEN', 'GEMINI_API_KEY']:
        secret_value = load_kaggle_secret(secret_name)
        if secret_value and secret_name not in os.environ:
            os.environ[secret_name] = secret_value

if IS_KAGGLE and PULL_REPO_ON_KAGGLE:
    kaggle_repo_root = Path('/kaggle/working') / REPO_DIR_NAME
    auth_repo_url = github_auth_url(REPO_GIT_URL, os.getenv('GITHUB_TOKEN'))

    if kaggle_repo_root.exists() and (kaggle_repo_root / '.git').exists():
        run_command('git', 'fetch', 'origin', REPO_BRANCH, cwd=kaggle_repo_root)
        run_command('git', 'checkout', REPO_BRANCH, cwd=kaggle_repo_root)
        run_command('git', 'pull', 'origin', REPO_BRANCH, cwd=kaggle_repo_root)
    else:
        run_command(
            'git', 'clone', '--branch', REPO_BRANCH, auth_repo_url, str(kaggle_repo_root),
            cwd='/kaggle/working'
        )
    ROOT = kaggle_repo_root
else:
    ROOT = STAGED_ROOT

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
importlib.invalidate_caches()

if INSTALL_PROJECT_REQUIREMENTS:
    run_command(sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', cwd=ROOT)
else:
    print('Skipping pip install.')

service_account_key_path = None
dvc_config_local_path = None
if SETUP_DVC_SERVICE_ACCOUNT:
    service_account_key_path = write_service_account_key(GDRIVE_SERVICE_ACCOUNT_SECRET, ROOT)
    if service_account_key_path is None:
        print(f'{GDRIVE_SERVICE_ACCOUNT_SECRET} not found or invalid; skipping DVC service-account setup.')
    else:
        dvc_config_local_path = configure_dvc_service_account(ROOT, service_account_key_path, remote_name=DVC_PULL_REMOTE)
        os.environ.setdefault('DVC_SITE_CACHE_DIR', str(ROOT / '.dvc' / 'site-cache'))
        if DVC_PULL_ON_KAGGLE:
            run_command('dvc', 'pull', '-r', DVC_PULL_REMOTE, *DVC_PULL_TARGETS, cwd=ROOT)
else:
    print('SETUP_DVC_SERVICE_ACCOUNT is False; skipping DVC auth bootstrap.')

# Import path constants dynamically after project root is setup
from src.config import CAFEF_DATA_DIR, MODELS_DATA_DIR, PROCESSED_DATA_DIR

TRAINING_INPUT = ROOT / 'data' / 'main' / 'cafef' / 'training_input.parquet'
TRAINING_OUTPUT = ROOT / 'data' / 'main' / 'cafef' / 'training_corpus.parquet'
BOOTSTRAP_OUTPUT = ROOT / 'data' / 'main' / 'cafef' / 'training_bootstrap_labels.parquet'
MERGED_LABELED_OUTPUT = ROOT / 'data' / 'main' / 'cafef' / 'training_labeled.parquet'
MODEL_DIR = ROOT / 'models' / 'phobert-sentiment' / 'latest'
CAFEF_ARTICLES = ROOT / 'data' / 'main' / 'processed' / 'articles_clean.parquet'
CAFEF_INPUT = ROOT / 'data' / 'main' / 'cafef' / 'cafef_input.parquet'
SENTIMENT_OUTPUT = ROOT / 'data' / 'main' / 'processed' / 'article_sentiment_scores.parquet'
DAILY_NEWS = ROOT / 'data' / 'main' / 'processed' / 'daily_news_prices.parquet'

BASE_MODEL = os.getenv('PHOBERT_BASE_MODEL', 'vinai/phobert-base-v2')
BATCH_SIZE = 16
MAX_LENGTH = 256
EPOCHS = 2
LEARNING_RATE = 2e-5
SEED = 42

# LLM Bootstrapping configurations
BOOTSTRAP_BACKEND = 'gemini'
BOOTSTRAP_MODEL = 'gemini-2.0-flash'
CONFIDENCE_THRESHOLD = 0.8

def run_module(module: str, *args: str) -> None:
    run_command(sys.executable, '-m', module, *args, cwd=ROOT)

for path in [TRAINING_OUTPUT.parent, MODEL_DIR.parent, SENTIMENT_OUTPUT.parent]:
    path.mkdir(parents=True, exist_ok=True)

print('ROOT                :', ROOT)
print('GITHUB_TOKEN loaded :', bool(os.getenv('GITHUB_TOKEN')))
print('GEMINI_API_KEY set  :', bool(os.getenv('GEMINI_API_KEY')))


## 3 - Prepare A Normalized Training Corpus

In [ ]:
run_module(
    'src.sentiment.prepare_training_data',
    '--input-file', str(TRAINING_INPUT),
    '--output-file', str(TRAINING_OUTPUT),
)
pd.read_parquet(TRAINING_OUTPUT).head(3)


## 4 - LLM Bootstrapping Stage (Ollama / Gemini)

Uses local or hosted language models to tag expected market-impact sentiment for training corpus articles. Make sure `GEMINI_API_KEY` is present in your environment variables if using the `gemini` backend.

In [ ]:
if BOOTSTRAP_BACKEND == 'gemini':
    assert os.getenv('GEMINI_API_KEY'), "GEMINI_API_KEY environment variable is required for Gemini bootstrapping."

run_module(
    'src.sentiment.bootstrap_labels',
    '--input-file', str(TRAINING_OUTPUT),
    '--output-file', str(BOOTSTRAP_OUTPUT),
    '--backend', BOOTSTRAP_BACKEND,
    '--model', BOOTSTRAP_MODEL,
    '--confidence-threshold', str(CONFIDENCE_THRESHOLD),
    '--limit', '50'  # Remove or increase this limit for the full dataset
)

# Merge LLM bootstrap annotations back into training corpus splits
run_module(
    'src.sentiment.merge_annotations',
    '--corpus-file', str(TRAINING_OUTPUT),
    '--annotations-file', str(BOOTSTRAP_OUTPUT),
    '--output-file', str(MERGED_LABELED_OUTPUT),
    '--confidence-threshold', str(CONFIDENCE_THRESHOLD),
    '--seed', str(SEED),
)

pd.read_parquet(MERGED_LABELED_OUTPUT)[['article_id', 'label', 'confidence', 'split']].head(5)


## 5 - Train The Classifier

In [ ]:
run_module(
    'src.sentiment.train_classifier',
    '--labeled-input', str(MERGED_LABELED_OUTPUT),
    '--output-dir', str(MODEL_DIR),
    '--base-model', str(BASE_MODEL),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--learning-rate', str(LEARNING_RATE),
    '--max-length', str(MAX_LENGTH),
    '--seed', str(SEED),
)
pd.read_json(MODEL_DIR / 'evaluation.json')


## 6 - Run CafeF Inference And Validation

In [ ]:
run_module(
    'src.sentiment.run_pipeline',
    '--mode', 'infer',
    '--model-dir', str(MODEL_DIR),
    '--cafef-input', str(CAFEF_ARTICLES),
    '--cafef-prepared-output', str(CAFEF_INPUT),
    '--sentiment-output', str(SENTIMENT_OUTPUT),
    '--daily-news-file', str(DAILY_NEWS),
    '--batch-size', str(BATCH_SIZE),
    '--max-length', str(MAX_LENGTH),
)
pd.read_parquet(SENTIMENT_OUTPUT).head(5)
